# Learning Buddy - uji pencarian + pelatihan adapter Pendidikan Pancasila

Notebook ini melakukan dua hal dalam satu kali jalan:
1. menguji pencarian website (Recursive, 3 potongan) dengan 1.014 soal PJOK Merdeka dan
   menghitung ambang pengecekan 1 (`hasil/`);
2. melatih 6 adapter LoRA **Pendidikan Pancasila** kelas 1-6 di atas Qwen2.5-7B-Instruct
   4-bit (Tabel 3.9), pembagian kanonik Pancasila 497/126/158 (Tabel 1.1).

Adapter **PJOK tidak dilatih ulang**: dipakai adapter G1 hasil chunking Recursive
(`D:\Skripsi\adapter-pjok\adapters_recursive`, Token F1 0,648 vs baseline 0,122), diunggah
sebagai dataset Kaggle **adapter-pjok**. Untuk melatih ulang PJOK juga, ubah MAPEL_LATIH.

Konteks latih dicari dengan layanan pencarian yang sama dengan saat menjawab (disaring
jenjang dan mata pelajaran) dan hanya disisipkan bila memuat
jawaban, sehingga adapter mengenal prompt tanpa konteks (S2) maupun berkonteks (S4).
Penyandi konteks: Qwen3-Embedding-4B di GPU kedua + FAISS + cross-encoder bge-reranker-v2-m3.

**Cara menjalankan**

1. Input: Dataset **learning-buddy-kode** (zip dari `python deploy/kemas.py`).
2. *Settings*: Accelerator **GPU T4 x2**, Internet **On**.
3. Tekan **Save Version > Save & Run All (Commit)**. Notebook berjalan sendiri di latar
   belakang (perkiraan 1,5-2 jam) dan browser boleh ditutup.
4. Setelah selesai, tab *Output* berisi `adapters/PKN/` dan `hasil/`. Notebook ini lalu
   ditambahkan sebagai input notebook server (*Add Input > Your Work*).

In [ ]:
# Salin kode ke folder kerja
import glob, os, shutil, subprocess, sys, time, re, json

# Matikan sisa server/tunnel dari jalankan sebelumnya. Proses latar itu TIDAK ikut mati saat
# kernel di-restart dan tetap memegang memori GPU (penyebab CUDA out of memory).
for nama in ('tb.server', 'cloudflared'):
    subprocess.run(['pkill', '-f', nama], check=False)
time.sleep(3)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
# indeks FAISS disimpan di luar folder kerja agar tidak dibangun ulang setiap Run All
os.environ.setdefault('TB_FAISS', '/kaggle/working/faiss_cache')
print(subprocess.run(['nvidia-smi', '--query-gpu=index,memory.used,memory.total', '--format=csv'],
                     capture_output=True, text=True).stdout)

# dicari di mana pun dalam input: zip, folder LearningBuddy, atau isi foldernya langsung
hit = glob.glob('/kaggle/input/**/tb/agen.py', recursive=True)
assert hit, 'Dataset learning-buddy-kode belum ditambahkan sebagai input.'
# Bila ada lebih dari satu salinan kode (dataset lama ikut terpasang), pakai yang terbaru:
# yang sudah punya tb/validasi_recursive.py, lalu yang berkasnya paling baru.
calon = [h.rsplit('/tb/', 1)[0] for h in hit]
calon.sort(key=lambda d: (os.path.exists(d + '/tb/validasi_recursive.py'),
                          os.path.getmtime(d + '/tb/server.py')), reverse=True)
if len(calon) > 1:
    print('PERINGATAN: ada', len(calon), 'salinan kode di input; dipakai yang terbaru.')
    print('  Sebaiknya lepas dataset kode lama dari Input:', calon[1:])
SUMBER = calon[0]
assert os.path.exists(SUMBER + '/data/potongan.jsonl'),     'Folder data/ tidak ikut terunggah; unggah ulang seluruh folder LearningBuddy.'
KERJA = '/kaggle/working/LearningBuddy'
shutil.rmtree(KERJA, ignore_errors=True)
shutil.copytree(SUMBER, KERJA)
os.chdir(KERJA)
sys.path.insert(0, KERJA)
print('kode:', SUMBER, '->', KERJA)

In [ ]:
# Pasang pustaka. Versi dicetak agar dapat dicatat di Bab IV.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt',
                '-r', 'requirements-gpu.txt'], check=True)
import importlib, importlib.metadata as md
importlib.invalidate_caches()
for p in ['torch', 'transformers', 'peft', 'bitsandbytes', 'accelerate',
          'sentence-transformers', 'faiss-cpu', 'langgraph', 'fastapi']:
    try: print(f'{p:22s}', md.version(p))
    except Exception: print(f'{p:22s} TIDAK ADA')
import torch
print('GPU:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# Indeks FAISS potongan Recursive (sudah dirapikan) per mapel per kelas, buku Kurikulum
# Merdeka, dengan penyandi pilihan (4.379 potongan, sekitar 5-8 menit).
# Ganti TB_MODEL_EMBEDDING bila hasil tb.uji_embedding memilih penyandi lain.
os.environ.setdefault('TB_MODEL_EMBEDDING', 'Qwen/Qwen3-Embedding-4B')
os.environ.setdefault('TB_DEVICE_CARI', 'cuda:1' if torch.cuda.device_count() > 1 else 'cuda')
print('penyandi', os.environ['TB_MODEL_EMBEDDING'], 'di', os.environ['TB_DEVICE_CARI'])
assert torch.cuda.device_count() > 1, 'Aktifkan Accelerator GPU T4 x2: penyandi 4B butuh GPU kedua'
from tb.pencarian import muat_potongan, bangun_indeks_faiss, folder_indeks, boleh_dicari, MODEL_EMBEDDING
ps = muat_potongan()
meta = os.path.join(folder_indeks(os.environ['TB_MODEL_EMBEDDING']), 'meta.json')
lama = json.load(open(meta)) if os.path.exists(meta) else {}
# dipakai ulang hanya bila indeks tersimpan dibangun dari potongan yang sama
if 'strategi' in lama and lama.get('potongan') == sum(boleh_dicari(p) for p in ps):
    print('indeks sudah ada, dipakai ulang:', lama)
else:
    bangun_indeks_faiss(ps, os.environ['TB_MODEL_EMBEDDING'])
import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Uji pipeline website: 1.014 soal PJOK Merdeka (Validasi_Soal_PJOK), dicari seperti saat
# website berjalan (Recursive, reranker, 3 potongan). Dibandingkan dengan tanpa reranker,
# masing-masing 1 dan 3 potongan. Sekaligus mengkalibrasi ambang pengecekan 1
# (ambang_relevan.json). Sekitar 10-20 menit.
from tb.evaluasi_pencarian import evaluasi
hasil_cari = evaluasi()
os.makedirs('/kaggle/working/hasil', exist_ok=True)
for f in ['evaluasi_pencarian.json', 'Evaluasi_Pencarian.xlsx', 'Validasi_Soal_PJOK_Recursive.xlsx']:
    shutil.copy(f'hasil/{f}', f'/kaggle/working/hasil/{f}')
shutil.copy(os.path.join(folder_indeks(os.environ['TB_MODEL_EMBEDDING']), 'ambang_relevan.json'),
            '/kaggle/working/hasil/ambang_relevan.json')
print('ambang pengecekan 1:', hasil_cari['ambang_relevan']['ambang'], hasil_cari['ambang_relevan']['kinerja'])
import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Pelatihan. Adapter yang sudah selesai dilewati, jadi sel ini aman dijalankan ulang.
MAPEL_LATIH = ('PKN',)          # PJOK memakai adapter G1 Recursive yang sudah ada
from tb.latih import latih
ringkasan = latih(MAPEL_LATIH, keluaran='/kaggle/working/adapters', cari='semantik')

In [ ]:
# Ringkasan untuk Bab IV, dan pemeriksaan bahwa adapter benar-benar ada
import pandas as pd
r = json.load(open('/kaggle/working/adapters/ringkasan_latih.json'))
display(pd.DataFrame(r['adapter']))
kurang = [f'{m} kelas {k}' for m in MAPEL_LATIH for k in range(1, 7)
          if not os.path.exists(f'/kaggle/working/adapters/{m}/grade_{k}/adapter_model.safetensors')]
print('SEMUA ADAPTER LENGKAP' if not kurang else f'BELUM ADA: {kurang}')
shutil.rmtree('/kaggle/working/LearningBuddy', ignore_errors=True)   # output cukup adapters/